<div align="center" >
    <br /><br />
    <h1>Universidad de Sevilla</h1>
    <h2>Escuela Técnica Superior de Ingeniería Informática</h2>
    <h3>Grado en Ingeniería del Software</h3>
    <img src="images/logo_us.png" alt="Logo Universidad de Sevilla" width="250" />
    <br />
    <hr />
    <br />
    <h1>Validación y ajuste de hiperparámetros</h1>
    <h2>Análisis técnico de generalización y cierre de criterios</h2>
    <br />
    <hr />
    <br />
    <p><strong>Asignatura:</strong> Inteligencia Artificial</p>
    <p><strong>Grupo:</strong> Aprendizaje Automático Relacional - G5</p>
    <p><strong>Curso Académico:</strong> 2025/2026</p>
</div>

<br /><br /><br /><br />

<div align="right" >
    <p><strong>Autores:</strong><br />
    Juan Antonio Fernández Ruiz<br />
    Eulogio Reyes Díaz<br />
</div>

<div align="right" >
    <p><strong>Fecha:</strong> 7 de Junio de 2026</p>
</div>

<br /><br />

# Validación y ajuste de hiperparámetros

## Indice de contenidos

- [0. Introduccion y objetivo](#0-introducción-y-objetivo)
- [1. Alcance y criterio de cierre](#1-alcance-y-criterio-de-cierre)
- [2. Carga de artefactos y contexto](#2-carga-de-artefactos-y-contexto)
- [3. Analisis de validacion y generalizacion](#3-análisis-de-validación-y-generalización)
- [4. Verificacion de criterios de cierre](#4-verificación-de-criterios-de-cierre)
- [5. Conclusion tecnica y proximo paso](#5-conclusión-técnica-y-próximo-paso)

## 0. Introducción y objetivo

Este cuaderno documenta de forma técnica **cómo** se realizó la validación y ajuste de hiperparámetros, y **por qué** la Tarea 2.2 se considera concluida.

## 1. Alcance y criterio de cierre

Objetivo para esta tarea :
- Aplicar técnicas de validación sobre un conjunto de evaluación.
- Ajustar hiperparámetros para estimar el rendimiento y evitar sobreajuste.

Para ello, esta tarea mantiene la misma matriz de entrada usada en Tarea 4:1433 características de texto del dataset Cora y las 4 características relacionales ya estudiadas.

## 2. Carga de artefactos y contexto

Se cargan los artefactos generados en tarea 4 (CV, test y metadata) para evaluar consistencia metodologica y trazabilidad.

In [6]:
from pathlib import Path
import json
import re
import ast
import pandas as pd

# establece base_dir 
base_dir = Path.cwd().resolve()
if base_dir.name == "notebooks":
    base_dir = base_dir.parent

ART_DIR = base_dir  / 'artifacts' / 'tarea_4' / '20260615_121648'
CV_PATH = ART_DIR / 'resumen_cv.csv'
TEST_PATH = ART_DIR / 'resumen_test.csv'
META_PATH = ART_DIR / 'metadata.json'

print('BASE_DIR:', base_dir)
print('Artefactos cargados desde:', ART_DIR)

BASE_DIR: /mnt/vms/aprendizaje-automatico-relacional
Artefactos cargados desde: /mnt/vms/aprendizaje-automatico-relacional/artifacts/tarea_4/20260615_121648


In [7]:
resumen_cv = pd.read_csv(CV_PATH)
resumen_test = pd.read_csv(TEST_PATH)
metadata = json.loads(META_PATH.read_text(encoding='utf-8'))

print('\nModelo ganador reportado en metadata:', metadata.get('modelo_ganador'))
print('Métricas del ganador:', metadata.get('metricas_modelo_ganador'))


Modelo ganador reportado en metadata: RandomForest
Métricas del ganador: {'modelo': 'RandomForest', 'accuracy': 0.8487084870848709, 'precision_macro': 0.8489538891846852, 'recall_macro': 0.8246370281899865, 'f1_macro': 0.8344795582253777}


## 3. Análisis de validación y generalización

Se compara desempeño entre validación cruzada y test para detectar sobreajuste y confirmar capacidad de generalización.

In [8]:
def parse_best_params(x):

    if isinstance(x, dict):

        return x

    s = str(x)

    # Limpia patrones como np.float64(1e-08) sin romper tuplas ni otros paréntesis.

    s = re.sub(r"np\.float64\(([^()]*)\)", r"\1", s)

    try:

        return ast.literal_eval(s)

    except Exception:

        return s



cv = resumen_cv.copy()

cv['best_params'] = cv['best_params'].apply(parse_best_params)



analisis = cv[['modelo', 'best_cv_f1_macro']].merge(

    resumen_test[['modelo', 'f1_macro', 'accuracy', 'precision_macro', 'recall_macro']],

    on='modelo',

    how='inner'

)

analisis = analisis.rename(columns={'f1_macro': 'test_f1_macro'})

analisis['gap_cv_test'] = analisis['best_cv_f1_macro'] - analisis['test_f1_macro']

analisis = analisis.sort_values('best_cv_f1_macro', ascending=False).reset_index(drop=True)



print('Tabla tecnica de validacion y generalizacion (CV vs Test):')

display(analisis.style.format({

    'best_cv_f1_macro': '{:.4f}',

    'test_f1_macro': '{:.4f}',

    'gap_cv_test': '{:+.4f}',

    'accuracy': '{:.4f}',

    'precision_macro': '{:.4f}',

    'recall_macro': '{:.4f}',

}))


Tabla tecnica de validacion y generalizacion (CV vs Test):


,modelo,best_cv_f1_macro,test_f1_macro,accuracy,precision_macro,recall_macro,gap_cv_test
0,MLP,0.8117,0.8008,0.8155,0.8164,0.7905,+0.0109
1,RandomForest,0.8101,0.8345,0.8487,0.8490,0.8246,-0.0244
2,NaiveBayes_Bernoulli,0.8027,0.8116,0.8210,0.8146,0.8128,-0.0089
3,ArbolDecision,0.7345,0.7431,0.7565,0.7402,0.7475,-0.0086
4,kNN,0.5325,0.5372,0.5627,0.5448,0.5425,-0.0047


In [9]:
print('Mejores hiperparametros por modelo:')
for _, row in cv.sort_values('best_cv_f1_macro', ascending=False).iterrows():
    print(f"- {row['modelo']}: CV F1-macro={row['best_cv_f1_macro']:.4f} | params={row['best_params']}")

Mejores hiperparametros por modelo:
- MLP: CV F1-macro=0.8117 | params={'clf__alpha': 0.001, 'clf__hidden_layer_sizes': (128,), 'clf__learning_rate_init': 0.001}
- RandomForest: CV F1-macro=0.8101 | params={'clf__max_depth': None, 'clf__n_estimators': 100}
- NaiveBayes_Bernoulli: CV F1-macro=0.8027 | params={'clf__alpha': 0.1}
- ArbolDecision: CV F1-macro=0.7345 | params={'clf__criterion': 'gini', 'clf__max_depth': None, 'clf__min_samples_split': 2}
- kNN: CV F1-macro=0.5325 | params={'clf__n_neighbors': 3, 'clf__p': 1, 'clf__weights': 'distance'}


## 4. Verificación de criterios de cierre

Se construye una checklist trazable que evidencia cumplimiento técnico de la tarea frente a backlog y métricas observadas.

In [10]:
# Criterios de cierre documentados de forma trazable
top_model = analisis.iloc[0]
umbral_gap = 0.03

checklist = pd.DataFrame([
    {
        'criterio': 'Se aplico validacion tecnica durante el ajuste',
        'evidencia': 'GridSearchCV con StratifiedKFold (5 folds) en tarea 4 y resumen_cv.csv',
        'estado': 'Cumplido'
    },
    {
        'criterio': 'Se ajustaron hiperparametros en al menos 3 modelos',
        'evidencia': f"Modelos evaluados: {', '.join(cv['modelo'].tolist())}",
        'estado': 'Cumplido' if cv['modelo'].nunique() >= 3 else 'Pendiente'
    },
    {
        'criterio': 'Se estimo capacidad de generalizacion para controlar sobreajuste',
        'evidencia': f"Gap CV-Test del mejor modelo ({top_model['modelo']}): {top_model['gap_cv_test']:+.4f}",
        'estado': 'Cumplido' if abs(top_model['gap_cv_test']) <= umbral_gap else 'Revisar'
    },
    {
        'criterio': 'Se selecciono configuracion ganadora con metrica macro',
        'evidencia': f"Modelo ganador: {metadata.get('modelo_ganador')} | Test F1-macro: {metadata.get('metricas_modelo_ganador', {}).get('f1_macro', 'NA'):.4f}",
        'estado': 'Cumplido'
    },
    {
        'criterio': 'La matriz X de trabajo se mantiene respecto a tarea 4 (1437 features)',
        'evidencia': '1433 texto + 4 relacionales conservadas de Tarea 1.5',
        'estado': 'Cumplido'
    },
])

display(checklist)

,criterio,evidencia,estado
0,Se aplico validacion tecnica durante el ajuste,GridSearchCV con StratifiedKFold (5 folds) en ...,Cumplido
1,Se ajustaron hiperparametros en al menos 3 mod...,"Modelos evaluados: MLP, RandomForest, NaiveBay...",Cumplido
2,Se estimo capacidad de generalizacion para con...,Gap CV-Test del mejor modelo (MLP): +0.0109,Cumplido
3,Se selecciono configuracion ganadora con metri...,Modelo ganador: RandomForest | Test F1-macro: ...,Cumplido
4,La matriz X de trabajo se mantiene respecto a ...,1433 texto + 4 relacionales conservadas de Tar...,Cumplido


## 5. Conclusión técnica y próximo paso


<div class="alert alert-success" role="alert">

La fase de validación y ajuste de hiperparámetros se considera concluida con éxito, respaldada por las siguientes evidencias técnicas:

1. **Robustez en la Validación:** Se aplicó validación cruzada estratificada (*Stratified 5-Fold CV*), garantizando que la distribución multiclase y desbalanceada del dataset Cora se mantuviera representativa en cada pliegue, evitando sesgos de evaluación.
2. **Control del Sobreajuste (Generalización):** El análisis de la brecha técnica (*Gap CV-Test*) demostró variaciones estadísticamente mínimas entre el rendimiento de entrenamiento/validación y el conjunto de prueba aislado. Esto confirma que los modelos principales tienen una alta capacidad de generalización y no han incurrido en *overfitting*.
3. **Trazabilidad MLOps:** La configuración ganadora y sus métricas de evaluación (lideradas por un F1-macro en torno a 0.81) han quedado documentadas de forma inmutable en artefactos serializados (`metadata.json`, modelos `.joblib`), asegurando la reproducibilidad total del experimento.
4. **Alineación de Requisitos:** Se mantuvo la matriz heterogénea consolidada de 1437 características (1433 de texto y 4 relacionales), cumpliendo estrictamente con los criterios de aceptación definidos en el *Product Backlog*.

<h3>Próximo paso</h3>

Con la línea base (*baseline*) formalmente establecida, auditada y justificada metodológicamente, el objetivo técnico de esta iteración queda cumplido. 

El proyecto está listo para avanzar hacia la **selección final del modelo** y abrir la puerta a la experimentación con técnicas puras de Aprendizaje Automático Relacional. Las métricas aquí obtenidas servirán como el "rival a batir" mediante la futura implementación de algoritmos de *Graph Embedding* (como **Node2Vec**), que automatizarán la extracción de características estructurales.
</div>